<a href="https://colab.research.google.com/github/sonu786786/Responsible_AI/blob/main/Lab_05/quest1%262.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kagglehub gensim nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 14.5 MB/s eta 0:00:00


In [5]:
import os, re, string, requests, nltk
import pandas as pd
import numpy as np
from gensim.models import KeyedVectors

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

STOP_WORDS = set(stopwords.words('english'))


# 1. DOWNLOAD DATASET FROM KAGGLE

os.environ["KAGGLE_API_TOKEN"] = "KGAT_155b95a19a48a3e19bfe45f85966a4a0"

import kagglehub
path = kagglehub.dataset_download("kritanjalijain/amazon-reviews")
print("Dataset path:", path)

# List files in the downloaded path
for f in os.listdir(path):
    print(f)

# 2. LOAD & REDUCE DATASET
#    We use only 5000 rows to save time

# The dataset has train.csv and test.csv; columns: polarity, title, text
df = pd.read_csv(os.path.join(path, "train.csv"), header=None,
                 names=["polarity", "title", "text"], nrows=5000)

print(f"Loaded {len(df)} rows")
print(df.head(3))

reviews = df["text"].dropna().tolist()

# ─────────────────────────────────────────
# Q1. PREPROCESSING
# ─────────────────────────────────────────
def preprocess(text: str) -> list[str]:
    """Full preprocessing pipeline → returns list of tokens."""
    # Step 1: Lowercase
    text = text.lower()
    # Step 2: Remove punctuation and numbers
    text = re.sub(r"[^a-z\s]", "", text)   # keep only letters and spaces
    # Step 3: Tokenize
    tokens = word_tokenize(text)
    # Step 4: Remove stopwords
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1]
    return tokens

tokenized_reviews = [preprocess(r) for r in reviews]
print("\nSample preprocessed review:")
print(tokenized_reviews[0])


# LOAD GOOGLE WORD2VEC MODEL
# Download from HuggingFace (binary format)
MODEL_PATH = "GoogleNews-vectors-negative300.bin"

if not os.path.exists(MODEL_PATH):
    print("Downloading Word2Vec model... (~1.6 GB, takes a few minutes)")
    # Direct download from a public mirror
    url = "https://huggingface.co/fse/word2vec-google-news-300/resolve/main/word2vec-google-news-300.gz"
    # Alternative: use gensim's downloader
    import gensim.downloader as api
    w2v_model = api.load("word2vec-google-news-300")
    print("Model loaded via gensim downloader!")
else:
    w2v_model = KeyedVectors.load_word2vec_format(MODEL_PATH, binary=True)
    print("Model loaded from disk!")

VECTOR_SIZE = 300


# SENTENCE REPRESENTATION
# Average of word vectors for each review

def sentence_vector(tokens: list[str], model) -> np.ndarray:
    """Return mean of word vectors for tokens present in model."""
    vectors = [model[w] for w in tokens if w in model]
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(VECTOR_SIZE)

sentence_vectors = np.array([sentence_vector(t, w2v_model) for t in tokenized_reviews])
print(f"\nSentence vectors shape: {sentence_vectors.shape}")
# → (5000, 300)  each review is now a 300-dim vector

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Using Colab cache for faster access to the 'amazon-reviews' dataset.
Dataset path: /kaggle/input/amazon-reviews
amazon_review_polarity_csv.tgz
train.csv
test.csv
Loaded 5000 rows
   polarity                                  title  \
0         2         Stuning even for the non-gamer   
1         2  The best soundtrack ever to anything.   
2         2                               Amazing!   

                                                text  
0  This sound track was beautiful! It paints the ...  
1  I'm reading a lot of reviews saying that this ...  
2  This soundtrack is my favorite music of all ti...  

Sample preprocessed review:
['sound', 'track', 'beautiful', 'paints', 'senery', 'mind', 'well', 'would', 'recomend', 'even', 'people', 'hate', 'vid', 'game', 'music', 'played', 'game', 'chrono', 'cross', 'games', 'ever', 'played', 'best', 'music', 'backs', 'away', 'crude', 'keyboarding', 'takes', 'fresher', 'step', 'grate', 'guitars', 'soulful', 'orchestras', 'would', 'impress', '

In [6]:
# ─────────────────────────────────────────
# Q2a. WORD ANALOGIES (manual examples)
# ─────────────────────────────────────────
print("\n─── Q2a: Custom Analogy Examples ───")

# good − bad + excellent = ?
result = w2v_model.most_similar(positive=["good", "excellent"], negative=["bad"], topn=5)
print("good − bad + excellent =", result[0][0])
print("Top 5:", result)

# delivery − late + fast = ?
result2 = w2v_model.most_similar(positive=["delivery", "fast"], negative=["late"], topn=5)
print("\ndelivery − late + fast =", result2[0][0])
print("Top 5:", result2)

# ─────────────────────────────────────────
# Q2b. ANALOGY DATASET EVALUATION
# w1 w2 w3 → predict w4  (w2 - w1 + w3 = w4)
# ─────────────────────────────────────────
print("\n─── Q2b: Analogy Dataset Evaluation ───")

# Download the analogy file
ANALOGY_URL = ("https://raw.githubusercontent.com/nicholas-leonard/"
               "word2vec/master/questions-words.txt")
ANALOGY_FILE = "questions-words.txt"

if not os.path.exists(ANALOGY_FILE):
    import urllib.request
    urllib.request.urlretrieve(ANALOGY_URL, ANALOGY_FILE)
    print("Downloaded analogy file.")

# ── Parse the analogy file ──
sections = {}
current_section = None
with open(ANALOGY_FILE, "r") as f:
    for line in f:
        line = line.strip()
        if line.startswith(":"):
            current_section = line[2:]
            sections[current_section] = []
        elif current_section and line:
            parts = line.lower().split()
            if len(parts) == 4:
                sections[current_section].append(parts)

print(f"Sections found: {list(sections.keys())[:5]} ...")
total_analogies = sum(len(v) for v in sections.values())
print(f"Total analogy pairs: {total_analogies}")

# ─────────────────────────────────────────
# OPTIMIZATION: Restrict search space
# ─────────────────────────────────────────
# The full model has ~3M words → slow.
# Strategy: only search within the model's MOST FREQUENT words
# (frequency-ranked vocabulary). We take top 50,000 words.
# This covers nearly all analogy answers while being much faster.

TOP_K_VOCAB = 50_000
# gensim KeyedVectors stores words in frequency order
restricted_vocab = set(list(w2v_model.key_to_index.keys())[:TOP_K_VOCAB])
print(f"\nRestricted search space to top {TOP_K_VOCAB:,} words")

def evaluate_analogy_section(section_name, pairs, model, vocab, max_samples=200):
    """
    Evaluate analogy: w2 - w1 + w3 = w4
    Uses restricted vocab to speed up search.
    """
    correct = 0
    total = 0
    skipped = 0

    for w1, w2, w3, w4 in pairs[:max_samples]:
        # Skip if any word not in model
        if not all(w in model for w in [w1, w2, w3, w4]):
            skipped += 1
            continue

        # most_similar with restrict_vocab for speed
        try:
            preds = model.most_similar(
                positive=[w2, w3],
                negative=[w1],
                restrict_vocab=TOP_K_VOCAB,
                topn=1
            )
            predicted = preds[0][0].lower()
            if predicted == w4:
                correct += 1
        except Exception:
            skipped += 1
            continue
        total += 1

    accuracy = correct / total if total > 0 else 0
    return correct, total, skipped, accuracy

# Evaluate all sections (limit samples per section for speed)
print("\n{'Section':<35} {'Correct':>8} {'Total':>8} {'Accuracy':>10}")
print("-" * 65)

overall_correct, overall_total = 0, 0
for section, pairs in sections.items():
    c, t, s, acc = evaluate_analogy_section(section, pairs, w2v_model,
                                             restricted_vocab, max_samples=100)
    overall_correct += c
    overall_total += t
    print(f"{section:<35} {c:>8} {t:>8} {acc:>9.1%}")

print("-" * 65)
overall_acc = overall_correct / overall_total if overall_total else 0
print(f"{'OVERALL':<35} {overall_correct:>8} {overall_total:>8} {overall_acc:>9.1%}")

# ─────────────────────────────────────────
# SHOW EXAMPLE PREDICTIONS
# ─────────────────────────────────────────
print("\n─── Sample Predictions ───")
sample_section = list(sections.keys())[0]
for w1, w2, w3, w4 in sections[sample_section][:5]:
    if all(w in w2v_model for w in [w1, w2, w3]):
        pred = w2v_model.most_similar(positive=[w2, w3], negative=[w1],
                                       restrict_vocab=TOP_K_VOCAB, topn=1)
        print(f"  {w2} - {w1} + {w3} = {pred[0][0]:<15}  (expected: {w4})")

print("\n✅ Done!")


─── Q2a: Custom Analogy Examples ───
good − bad + excellent = terrific
Top 5: [('terrific', 0.6986191272735596), ('superb', 0.6440028548240662), ('fantastic', 0.6409966349601746), ('great', 0.6319711208343506), ('exceptional', 0.5893022418022156)]

delivery − late + fast = delievery
Top 5: [('delievery', 0.46103477478027344), ('twitch_muscles', 0.42857083678245544), ('Delivery', 0.4249102771282196), ('Eat##Hours.com', 0.39901983737945557), ('www.BobBruss.com', 0.39756283164024353)]

─── Q2b: Analogy Dataset Evaluation ───
Sections found: ['capital-common-countries', 'capital-world', 'currency', 'city-in-state', 'family'] ...
Total analogy pairs: 19544

Restricted search space to top 50,000 words

{'Section':<35} {'Correct':>8} {'Total':>8} {'Accuracy':>10}
-----------------------------------------------------------------
capital-common-countries                   7       85      8.2%
capital-world                              2       31      6.5%
currency                              